# SaaS/E-Commerce Cohort Retention & CLTV Analysis
## Master Notebook — Full Project Integration (Weeks 1–4)

**Organization:** Infotact Solutions  
**Program:** Data Analytics Internship — Project 2  
**Dataset:** Online Retail Dataset (Kaggle)  
**Link:** https://www.kaggle.com/datasets/vijayuv/onlineretail  
**Analysis Period:** December 2010 — December 2011  
**Date:** 3rd July 2026  

> ⚠️ Raw data files are excluded from GitHub per data privacy guidelines.  
> Download the dataset from the Kaggle link above and place it in a local folder to run this notebook.

---

## 📋 Project Overview
This master notebook consolidates all 4 weeks of analysis into one clean end-to-end pipeline:

| Week | Focus | Key Output |
|---|---|---|
| Week 1 | Data Cleaning & Validation | 392,692 clean transactions from 541,909 raw records |
| Week 2 | Cohort Retention Matrix | 86.70% Month 1 churn identified |
| Week 3 | CLTV Calculation | Mean CLTV £2,048 per customer |
| Week 4 | Visualization & Strategic Insights | Cohort retention heatmap |

---

## Step 1: Import All Libraries
Importing all required libraries for the complete analysis pipeline.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Seaborn version: {sns.__version__}")

---
# 📊 WEEK 1: Transactional Data Cleaning & Wrangling
**Date Range:** 5th – 11th June 2026  
**Objective:** Prepare the raw transactional dataset for cohort analysis by removing
invalid transactions, handling missing values, validating data types and calculating
CohortMonth for every unique customer.

---

## Step 2: Load Raw Dataset
Loading the raw Online Retail Dataset.  
Original dataset contains **541,909 rows** across 8 columns.

In [ ]:
# Load raw dataset
df = pd.read_csv("OnlineRetail.csv", encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample Data:")
print(df.head())
print(f"\nData Types:")
print(df.dtypes)

## Step 3: Dataset Overview & Initial Inspection
Before cleaning, we inspect the dataset structure and identify potential data quality issues
including missing values, duplicates and invalid records.

In [ ]:
# Check missing values
print("Missing Values per Column:")
print(df.isnull().sum())

# Check duplicates
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

# Check dataset shape
print(f"\nDataset Shape: {df.shape}")
print(f"\nNumerical Summary:")
print(df.describe())

## Step 4: Remove Cancelled Transactions
Invoices beginning with the letter 'C' represent cancelled orders.
These records do not indicate successful purchases and must be excluded
from customer purchase analysis.

In [ ]:
original_shape = df.shape

# Remove cancelled transactions (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

print(f"Original Shape: {original_shape}")
print(f"After Removing Cancelled Transactions: {df.shape}")
print(f"Cancelled Transactions Removed: {original_shape[0] - df.shape[0]}")
print(f"\nSample InvoiceNo values: {df['InvoiceNo'].head().tolist()}")

## Step 5: Remove Missing Customer IDs
Customer retention and cohort analysis require unique customer identification.
Rows containing missing CustomerID values are removed because they cannot
be associated with a specific customer.

In [ ]:
before = df.shape[0]

# Remove missing CustomerIDs
df_cleaned = df.dropna(subset=['CustomerID'])

print(f"Before removing missing CustomerIDs: {before}")
print(f"After removing missing CustomerIDs: {df_cleaned.shape[0]}")
print(f"Rows removed: {before - df_cleaned.shape[0]}")

## Step 6: Missing Value Analysis
Identifying all remaining columns that contain null values
and measuring the extent of missing information.

In [ ]:
print("Missing values after CustomerID removal:")
print(df_cleaned.isnull().sum())

# Drop any remaining nulls
df_cleaned = df_cleaned.dropna()
print(f"\nAfter dropping all nulls: {df_cleaned.shape}")

## Step 7: Remove Duplicate Rows
Removing duplicate rows to avoid redundant records and ensure
accurate data assessment and analysis.

In [ ]:
print(f"Duplicate Rows Before: {df_cleaned.duplicated().sum()}")

# Remove duplicates
df_cleaned = df_cleaned.drop_duplicates()

print(f"Duplicate Rows After: {df_cleaned.duplicated().sum()}")
print(f"Dataset Shape After Deduplication: {df_cleaned.shape}")

## Step 8: Validate & Convert Data Types
Ensuring all columns have correct data types for analysis:
- InvoiceDate → datetime
- Quantity → numeric (int64)
- UnitPrice → numeric (float64)
- CustomerID → int64 then string

In [ ]:
print("Initial Data Types:")
print(df_cleaned.dtypes)

# Convert InvoiceDate to datetime
df_cleaned["InvoiceDate"] = pd.to_datetime(df_cleaned["InvoiceDate"])

# Convert Quantity to numeric
df_cleaned["Quantity"] = pd.to_numeric(df_cleaned["Quantity"], errors="coerce")

# Convert UnitPrice to numeric
df_cleaned["UnitPrice"] = pd.to_numeric(df_cleaned["UnitPrice"], errors="coerce")

# Convert CustomerID to int64 then string
df_cleaned["CustomerID"] = df_cleaned["CustomerID"].astype("int64").astype("str")

print("\nAfter Validation — Data Types:")
print(df_cleaned.dtypes)

## Step 9: Remove Invalid Quantities & Prices
Transactions with negative quantities or prices are removed
to retain only valid purchase records for analysis.

### Checks Performed:
- Negative Quantity — invalid purchase records
- Negative UnitPrice — invalid price records

In [ ]:
print(f"Negative Quantity Records: {(df_cleaned['Quantity'] < 0).sum()}")
print(f"Negative UnitPrice Records: {(df_cleaned['UnitPrice'] < 0).sum()}")

# Remove invalid records
df_cleaned = df_cleaned[df_cleaned["Quantity"] > 0]
df_cleaned = df_cleaned[df_cleaned["UnitPrice"] > 0]

print(f"\nFinal Shape After Removing Invalid Records: {df_cleaned.shape}")

## Step 10: Extract Transaction Month
The transaction month is extracted from the InvoiceDate column.
This feature is required for cohort analysis and customer retention tracking.

In [ ]:
# Extract Transaction Month
df_cleaned['TransactionMonth'] = df_cleaned['InvoiceDate'].dt.to_period('M')

print("TransactionMonth extracted successfully")
print(df_cleaned[['InvoiceDate', 'TransactionMonth']].head())

## Step 11: Calculate Customer Cohort Month
The first purchase month of each customer is identified.
This month becomes the customer's Cohort Month and serves
as the basis for all cohort analysis.

In [ ]:
# Calculate CohortMonth (first purchase month per customer)
cohort_month = (
    df_cleaned.groupby('CustomerID')['InvoiceDate']
    .min()
    .dt.to_period('M')
)

# Merge back into dataset
df_cleaned['CohortMonth'] = df_cleaned['CustomerID'].map(cohort_month)

print("CohortMonth calculated and merged successfully")
print(df_cleaned[['CustomerID', 'InvoiceDate', 'TransactionMonth', 'CohortMonth']].head(20))

## Step 12: Week 1 Summary & Validation
Final validation of the cleaned dataset before proceeding to Week 2.

In [ ]:
print("=" * 50)
print("WEEK 1 CLEANING SUMMARY")
print("=" * 50)
print(f"Original Records: 541,909")
print(f"Final Clean Records: {df_cleaned.shape[0]}")
print(f"Records Removed: {541909 - df_cleaned.shape[0]}")
print(f"Unique Customers: {df_cleaned['CustomerID'].nunique()}")
print(f"Date Range: {df_cleaned['InvoiceDate'].min()} to {df_cleaned['InvoiceDate'].max()}")
print(f"Unique Countries: {df_cleaned['Country'].nunique()}")
print(f"Unique Products: {df_cleaned['StockCode'].nunique()}")
print(f"\nFinal Columns: {df_cleaned.columns.tolist()}")

### 💡 Week 1 Business Implication:
> Over **27% of raw transactions were invalid** — cancelled, missing customer data,
> or containing negative values. This highlights the critical importance of data
> validation before making any business decisions from raw transactional data.
> A business making decisions on uncleaned data risks misallocating significant resources.

---

---
# 📊 WEEK 2: Building the Cohort Retention Matrix
**Date Range:** 12th – 18th June 2026  
**Objective:** Build absolute and percentage retention matrices to understand 
how many customers return each month after their first purchase.

---

## Step 13: Cohort Grouping
Grouping customers by CohortMonth and CohortIndex to count
unique customers active in each time period.

| Column | Description |
|---|---|
| CohortMonth | Month of customer's first purchase |
| CohortIndex | Number of months since first purchase |

In [ ]:
# Calculate CohortIndex
df_cleaned['CohortIndex'] = (
    (df_cleaned['TransactionMonth'].dt.year - df_cleaned['CohortMonth'].dt.year) * 12 +
    (df_cleaned['TransactionMonth'].dt.month - df_cleaned['CohortMonth'].dt.month)
)

# Group by CohortMonth and CohortIndex
cohort_data = (
    df_cleaned.groupby(['CohortMonth', 'CohortIndex'])['CustomerID']
    .nunique()
    .reset_index()
)

print(f"Cohort Groups Shape: {cohort_data.shape}")
print(f"\nSample Cohort Data:")
print(cohort_data.head(10))

## Step 14: Build Absolute Retention Matrix
Using Pandas pivot_table to reshape the cohort data into a matrix.

| Axis | Represents |
|---|---|
| Rows | CohortMonth — acquisition month |
| Columns | Month 0, Month 1, Month 2 etc. |
| Values | Absolute number of retained customers |

> **Month 0** represents 100% of the original cohort size.

In [ ]:
# Build pivot table
retention_matrix = cohort_data.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='CustomerID'
)

# Format columns and index
retention_matrix.columns = [f"Month {int(col)}" for col in retention_matrix.columns]
retention_matrix.index = retention_matrix.index.astype(str)

# Validate Month 0
if retention_matrix["Month 0"].isnull().sum() == 0:
    print("✅ Validation Passed: Month 0 contains full cohort size for every cohort")
else:
    print("❌ Validation Failed: Some cohorts have missing Month 0 values")

print(f"\nRetention Matrix Shape: {retention_matrix.shape}")
print(f"\nAbsolute Retention Matrix:")
print(retention_matrix)

## Step 15: Verify Absolute Retained Users
Verifying and displaying the absolute number of retained users
for each Month 0, Month 1, Month 2 etc. across all cohorts.

In [ ]:
print("Month 0 — Original Cohort Sizes:")
print(retention_matrix["Month 0"])

print("\nTotal Retained Users Per Month (All Cohorts Combined):")
print(retention_matrix.sum())

# Validate no month exceeds Month 0
valid = (retention_matrix.drop(columns=["Month 0"])
         .le(retention_matrix["Month 0"], axis=0)
         .all().all())

print(f"\nMatrix Valid (no month exceeds Month 0): {valid}")

# Summary statistics
print("\nRetained Users Summary Statistics:")
print(retention_matrix.describe().round(2))

## Step 16: Calculate Percentage Retention Rate
Converting absolute numbers to percentage retention rates.

### Formula:
> **Retention % = (Month N Retained Users / Month 0 Users) x 100**

In [ ]:
# Calculate percentage retention
cohort_sizes = retention_matrix['Month 0']
percentage_matrix = retention_matrix.divide(cohort_sizes, axis=0) * 100

# Validate Month 0 is 100%
print("Month 0 Retention (should all be 100%):")
print(percentage_matrix["Month 0"])

valid = (percentage_matrix.fillna(0) <= 100).all().all()
print(f"\nAll values within 0-100%: {valid}")

print("\nPercentage Retention Matrix:")
print(percentage_matrix.round(2))

## Step 17: Key Retention Findings
Extracting key metrics to understand retention patterns across all cohorts.

In [ ]:
# Average retention per month
avg_retention = percentage_matrix.mean()
print("Average Retention Rate by Month (%):")
print(avg_retention.round(2))

# Month on month drop off
churn = avg_retention.diff().dropna()
highest_churn_month = churn.idxmin()
print(f"\nMonth with Highest Churn: {highest_churn_month}")

# Best and worst cohorts
print(f"\nBest Retention Cohort (Month 1): {percentage_matrix['Month 1'].idxmax()} — {percentage_matrix['Month 1'].max():.2f}%")
print(f"Worst Retention Cohort (Month 1): {percentage_matrix['Month 1'].idxmin()} — {percentage_matrix['Month 1'].min():.2f}%")
print(f"Overall Average Month 1 Retention: {percentage_matrix['Month 1'].mean():.2f}%")
print(f"Month 12 Average Retention: {percentage_matrix['Month 12'].mean():.2f}%")

### 💡 Week 2 Business Implications:

**Finding 1 — Massive Early Churn:**
> **79.38% of customers do not return after their first purchase.**
> Only 20.62% make a second purchase in Month 1.
> This is the single biggest retention problem for this business.

**Finding 2 — Seasonal Cohort Performance:**
> The **December 2010 cohort** performed best with 36.61% Month 1 retention = driven by holiday Shopping behaviour.
> The **November 2011 cohort** performed worst at just 11.15%.

**Finding 3 — Long-Term Loyal Customers:**
> Despite massive early churn, customers who survive past Month 3
> show **increasing** retention — reaching **26.55% by Month 12.**

---

---
# 📊 WEEK 3: Customer Lifetime Value (CLTV) Calculation
**Date Range:** 24th – 29th June 2026  
**Objective:** Calculate Historical CLTV for each customer and cohort segment
using AOV and Purchase Frequency.

### Core Formula:
> **CLTV = AOV x Purchase Frequency**

---

## Step 18: Calculate TotalRevenue
Adding TotalRevenue column to capture the monetary value
of each individual transaction.

> **TotalRevenue = Quantity x UnitPrice**

In [ ]:
# Calculate TotalRevenue
df_cleaned['TotalRevenue'] = df_cleaned['Quantity'] * df_cleaned['UnitPrice']

print("✅ TotalRevenue column added successfully")
print(df_cleaned[['CustomerID', 'InvoiceNo', 'Quantity', 'UnitPrice', 'TotalRevenue']].head())
print(f"\nTotal Revenue Summary:")
print(df_cleaned['TotalRevenue'].describe().round(2))

## Step 19: Cohort-Based Revenue Segmentation
Segmenting customer transactions by acquisition cohort and
calculating the total revenue generated by each customer within each cohort.

In [ ]:
# Customer cohort revenue
customer_cohort_revenue = (
    df_cleaned.groupby(['CohortMonth', 'CustomerID'])['TotalRevenue']
    .sum()
    .reset_index()
)

print(f"Customer Cohort Revenue Shape: {customer_cohort_revenue.shape}")
print(customer_cohort_revenue.head())

# Validate
print(f"\nMissing Values: {customer_cohort_revenue.isnull().sum().sum()}")
print(f"Duplicate Customer-Cohort Combinations: {customer_cohort_revenue.duplicated(subset=['CohortMonth', 'CustomerID']).sum()}")
print(f"\nRevenue Summary:")
print(customer_cohort_revenue['TotalRevenue'].describe().round(2))

## Step 20: Calculate Average Order Value (AOV)
AOV measures how much a customer spends on average per order.

> **AOV = Total Customer Revenue / Total Unique Orders per Customer**

In [ ]:
# Total revenue per customer
customer_revenue = (
    df_cleaned.groupby('CustomerID')['TotalRevenue']
    .sum()
    .reset_index()
    .rename(columns={'TotalRevenue': 'CustomerRevenue'})
)

# Total unique orders per customer
customer_orders = (
    df_cleaned.groupby('CustomerID')['InvoiceNo']
    .nunique()
    .reset_index()
    .rename(columns={'InvoiceNo': 'TotalOrders'})
)

# Calculate AOV
aov = customer_revenue.merge(customer_orders, on='CustomerID')
aov['AOV'] = aov['CustomerRevenue'] / aov['TotalOrders']

print(f"AOV Shape: {aov.shape}")
print(f"\nAOV Summary Statistics:")
print(aov['AOV'].describe().round(2))
print(f"\nSample AOV Output:")
print(aov.head())

# Validate AOV
print(f"\nMissing AOV Values: {aov['AOV'].isnull().sum()}")
print(f"Duplicate CustomerIDs: {aov['CustomerID'].duplicated().sum()}")

## Step 21: Calculate Purchase Frequency
Purchase Frequency measures how often a customer makes a purchase.

> **Purchase Frequency = Total Unique Invoices per Customer**

In [ ]:
# Calculate Purchase Frequency
purchase_freq = (
    df_cleaned.groupby('CustomerID')['InvoiceNo']
    .nunique()
    .reset_index()
)
purchase_freq.columns = ['CustomerID', 'PurchaseFrequency']

# Add frequency segments
def frequency_segment(value):
    if value == 1:
        return 'Low Frequency'
    elif 2 <= value <= 4:
        return 'Medium Frequency'
    else:
        return 'High Frequency'

purchase_freq['FrequencySegment'] = purchase_freq['PurchaseFrequency'].apply(frequency_segment)

print(f"Purchase Frequency Shape: {purchase_freq.shape}")
print(f"\nFrequency Segment Distribution:")
print(purchase_freq['FrequencySegment'].value_counts())
print(f"\nPurchase Frequency Summary:")
print(purchase_freq['PurchaseFrequency'].describe().round(2))

# Validate
print(f"\nMissing Values: {purchase_freq.isnull().sum().sum()}")
print(f"Duplicate CustomerIDs: {purchase_freq['CustomerID'].duplicated().sum()}")

## Step 22: Calculate Historical CLTV ⭐
Merging AOV and Purchase Frequency to compute the Historical CLTV
for each customer.

> **CLTV = AOV x Purchase Frequency**

In [ ]:
# Merge AOV and Purchase Frequency
cltv_df = aov.merge(purchase_freq, on='CustomerID')

# Calculate Historical CLTV
cltv_df['CLTV'] = cltv_df['AOV'] * cltv_df['PurchaseFrequency']

print(f"CLTV DataFrame Shape: {cltv_df.shape}")
print(f"\nSample CLTV Output:")
print(cltv_df.head(10))
print(f"\nCLTV Summary Statistics:")
print(cltv_df['CLTV'].describe().round(2))

## Step 23: Project 12-Month CLTV Per Cohort Segment
Scaling the historical CLTV to an annual projection and grouping
results by CohortMonth segment.

> **12-Month CLTV = Historical CLTV x 12**

In [ ]:
# Merge CohortMonth into cltv_df
cltv_df = cltv_df.merge(
    df_cleaned[['CustomerID', 'CohortMonth']].drop_duplicates(),
    on='CustomerID',
    how='left'
)

# Project 12-Month CLTV
cltv_df['CLTV_12Month'] = cltv_df['CLTV'] * 12

# Group by CohortMonth segment
cohort_cltv = cltv_df.groupby('CohortMonth').agg(
    Total_Customers=('CustomerID', 'count'),
    Avg_AOV=('AOV', 'mean'),
    Avg_Purchase_Frequency=('PurchaseFrequency', 'mean'),
    Avg_CLTV=('CLTV', 'mean'),
    Avg_12Month_CLTV=('CLTV_12Month', 'mean'),
    Total_12Month_CLTV=('CLTV_12Month', 'sum')
).round(2).reset_index()

print("12-Month CLTV Per Cohort Segment:")
print(cohort_cltv)
print(f"\n🏆 Best Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmax()])
print(f"\n⚠️ Worst Performing Cohort:")
print(cohort_cltv.loc[cohort_cltv['Avg_12Month_CLTV'].idxmin()])

## Step 24: Customer Segmentation
Segmenting customers into three value tiers based on
their 12-Month CLTV using percentile thresholds.

| Segment | Threshold |
|---|---|
| High Value | Top 25% (above 75th percentile) |
| Mid Value | Middle 50% |
| Low Value | Bottom 25% (below 25th percentile) |

In [ ]:
# Define thresholds
high_threshold = cltv_df['CLTV_12Month'].quantile(0.75)
low_threshold = cltv_df['CLTV_12Month'].quantile(0.25)

def segment_customer(cltv):
    if cltv >= high_threshold:
        return 'High Value'
    elif cltv <= low_threshold:
        return 'Low Value'
    else:
        return 'Mid Value'

cltv_df['Segment'] = cltv_df['CLTV_12Month'].apply(segment_customer)

print("Customer Segment Distribution:")
print(cltv_df['Segment'].value_counts())
print(f"\nHigh Value Threshold: ${high_threshold:,.2f}")
print(f"Low Value Threshold: ${low_threshold:,.2f}")
print(f"\nCLTV Summary by Segment:")
print(cltv_df.groupby('Segment')['CLTV_12Month'].mean().round(2))

## Step 25: Calculate Maximum Acceptable CAC
CAC should not exceed 30% of the projected 12-Month CLTV
to ensure business profitability.

> **Max CAC = CLTV_12Month x 0.30**

In [ ]:
# Calculate Max CAC
cltv_df['Max_CAC'] = cltv_df['CLTV_12Month'] * 0.30

print("CAC Summary by Segment:")
print(cltv_df.groupby('Segment')['Max_CAC'].mean().round(2))
print(f"\nOverall Average Max CAC: ${cltv_df['Max_CAC'].mean():,.2f}")
print(f"\nCLTV Gap between Best and Worst Cohort:")
best = cohort_cltv['Avg_12Month_CLTV'].max()
worst = cohort_cltv['Avg_12Month_CLTV'].min()
print(f"${best - worst:,.2f}")

## Step 26: Geographic CLTV Analysis
Analysing CLTV performance across different countries
to identify highest value geographic segments.

In [ ]:
# Geographic CLTV Analysis
country_cltv = df_cleaned.merge(cltv_df, on='CustomerID')

print("Top 10 Countries by Average CLTV:")
print(country_cltv.groupby('Country')['CLTV'].mean()
      .sort_values(ascending=False)
      .head(10).round(2))

# Cross validation
original_total = df_cleaned['TotalRevenue'].sum()
grouped_total = customer_cohort_revenue['TotalRevenue'].sum()
print(f"\nRevenue Cross-Validation:")
print(f"Original Total Revenue: ${original_total:,.2f}")
print(f"Grouped Total Revenue: ${grouped_total:,.2f}")
print(f"Match: {round(original_total) == round(grouped_total)}")

### 💡 Week 3 Business Implications:

**Finding 1 — High Value Customer Concentration:**
> Only **25% of customers (1,085)** fall into the High Value segment
> but generate an average 12-Month CLTV of **\$77,835** — nearly 9x
> more than Mid Value customers. These customers are critical to
> protect and retain at all costs.

**Finding 2 — Geographic Concentration:**
> **Netherlands (\$246,720)** and **EIRE (\$135,489)** customers generate
> significantly higher CLTV than the United Kingdom (\$7,780)
> despite the UK having far more customers.
> International customers represent a high-value but underutilised
> market segment with massive growth potential.

**Finding 3 — CAC Efficiency by Segment:**
> High Value customers justify a maximum CAC of **\$23,350.**
> Low Value customers only justify a maximum CAC of **\$640.**
> Mixing these budgets leads to significant marketing waste.

---
# 📊 WEEK 4: Visualization & Strategic Insights
**Date Range:** 30th June – 4th July 2026  
**Objective:** Visualize retention trends using heatmaps, decay charts
and revenue analysis to communicate findings to business stakeholders.

### Visualizations Produced:
1. Cohort Retention Heatmap
2. Retention Decay Line Charts
3. Total Revenue by Cohort Bar Chart
4. Top Countries by CLTV Bar Chart
5. Customer Segment Distribution Chart
6. AOV by Cohort Bar Chart

---

## Step 27: Cohort Retention Heatmap ⭐
Visualizing the percentage retention matrix as a colour-coded heatmap
using Seaborn. Darker cells = higher retention. Lighter cells = lower retention.

### Key Expected Findings:
- Darkest cells — Month 0 (100% retention — original cohort)
- Lightest cells — Month 1 to Month 3 (early churn period)
- Best cohort: December 2010 (36.61% Month 1 retention)
- Worst cohort: November 2011 (11.15% Month 1 retention)

In [ ]:
plt.figure(figsize=(16, 10))
sns.heatmap(
    percentage_matrix,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    linewidths=0.5,
    vmin=0,
    vmax=100
)
plt.title('Cohort Retention Rate Heatmap\nSaaS/E-Commerce Analysis — Infotact Solutions',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Months Since First Purchase', fontsize=12)
plt.ylabel('Cohort Month', fontsize=12)
plt.xticks(
    range(len(percentage_matrix.columns)),
    [f"Month {i}" for i in range(len(percentage_matrix.columns))],
    rotation=45
)
plt.tight_layout()

# Save as high DPI image
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/cohort_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Heatmap saved as outputs/cohort_retention_heatmap.png")

## Step 28: Retention Decay Line Charts ⭐
Plotting retention decay curves showing how each cohort's retention
declines from Month 0 to final month. Multiple cohorts overlaid
for side-by-side comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Chart 1 - All cohorts overlaid
ax1 = axes[0]
colors = plt.cm.tab20.colors
for i, cohort in enumerate(percentage_matrix.index):
    cohort_data = percentage_matrix.loc[cohort].dropna()
    months = range(len(cohort_data))
    ax1.plot(months, cohort_data.values,
             marker='o', markersize=4,
             color=colors[i % len(colors)],
             label=str(cohort), linewidth=1.5, alpha=0.8)

ax1.set_title('Retention Decay Curves — All Cohorts\n(Month 0 to Month 12)',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Months Since First Purchase', fontsize=11)
ax1.set_ylabel('Retention Rate (%)', fontsize=11)
ax1.set_xticks(range(13))
ax1.set_xticklabels([f'Month {i}' for i in range(13)], rotation=45)
ax1.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, title='Cohort Month')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=percentage_matrix['Month 1'].mean(), color='red',
            linestyle='--', linewidth=1.5)

# Chart 2 - Average retention decay
ax2 = axes[1]
avg_decay = percentage_matrix.mean()
months = range(len(avg_decay))
ax2.plot(months, avg_decay.values,
         marker='o', markersize=8,
         color='#1F4E79', linewidth=2.5, label='Average Retention')
ax2.fill_between(months, avg_decay.values, alpha=0.2, color='#1F4E79')

# Annotate key points
for i, (m, v) in enumerate(zip(months, avg_decay.values)):
    if i in [0, 1, 6, 12]:
        ax2.annotate(f'{v:.1f}%',
                     xy=(m, v), xytext=(5, 10),
                     textcoords='offset points',
                     fontsize=9, fontweight='bold',
                     color='#1F4E79')

ax2.set_title('Average Retention Decay Curve\n(Across All Cohorts)',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Months Since First Purchase', fontsize=11)
ax2.set_ylabel('Average Retention Rate (%)', fontsize=11)
ax2.set_xticks(months)
ax2.set_xticklabels([f'Month {i}' for i in months], rotation=45)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.suptitle('Customer Retention Decay Analysis — Infotact Solutions',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/02_retention_decay_charts.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Retention decay charts saved as outputs/02_retention_decay_charts.png")

## Step 29: Total Revenue by Cohort Bar Chart
Comparing total revenue generated by each acquisition cohort
to identify which cohorts are most valuable to the business.

In [ ]:
# Calculate total revenue by cohort
cohort_revenue = (
    df_cleaned.groupby('CohortMonth')['TotalRevenue']
    .sum()
    .reset_index()
    .sort_values('TotalRevenue', ascending=False)
)
cohort_revenue['CohortMonth'] = cohort_revenue['CohortMonth'].astype(str)

plt.figure(figsize=(16, 7))
bars = plt.bar(
    cohort_revenue['CohortMonth'],
    cohort_revenue['TotalRevenue'],
    color='#1F4E79',
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85
)

# Highlight top cohort
bars[0].set_color('#C00000')

plt.title('Total Revenue by Acquisition Cohort\nSaaS/E-Commerce Analysis — Infotact Solutions',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cohort Month (First Purchase Month)', fontsize=11)
plt.ylabel('Total Revenue ($)', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/03_revenue_by_cohort.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Revenue by cohort chart saved as outputs/03_revenue_by_cohort.png")

## Step 30: Top Countries by Average CLTV Bar Chart
Visualizing the geographic distribution of customer lifetime value
to identify the most valuable international markets.

In [ ]:
# Top 10 countries by average CLTV
top10 = (country_cltv.groupby('Country')['CLTV']
         .mean()
         .sort_values(ascending=True)
         .tail(10))

colors_bar = ['#C00000' if c == top10.index[-1] else '#1F4E79' for c in top10.index]

plt.figure(figsize=(14, 7))
bars = plt.barh(top10.index, top10.values,
                color=colors_bar, edgecolor='white',
                linewidth=0.5, alpha=0.85)

# Add value labels
for bar, val in zip(bars, top10.values):
    plt.text(val + 1000, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')

plt.title('Top 10 Countries by Average CLTV\nSaaS/E-Commerce Analysis — Infotact Solutions',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average CLTV ($)', fontsize=11)
plt.ylabel('Country', fontsize=11)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/04_top_countries_cltv.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Top countries CLTV chart saved as outputs/04_top_countries_cltv.png")

## Step 31: Customer Segment Distribution Chart
Visualizing the distribution of customers across
High, Mid and Low value segments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Pie chart
segment_counts = cltv_df['Segment'].value_counts()
colors_pie = ['#C00000', '#1F4E79', '#2E75B6']
explode = (0.05, 0, 0)

axes[0].pie(segment_counts.values,
            labels=segment_counts.index,
            autopct='%1.1f%%',
            colors=colors_pie,
            explode=explode,
            startangle=90,
            textprops={'fontsize': 11})
axes[0].set_title('Customer Segment Distribution\n(By Count)', fontsize=13, fontweight='bold')

# Bar chart - average CLTV by segment
segment_cltv = cltv_df.groupby('Segment')['CLTV_12Month'].mean().sort_values(ascending=False)
bar_colors = ['#C00000', '#1F4E79', '#2E75B6']

bars = axes[1].bar(segment_cltv.index, segment_cltv.values,
                    color=bar_colors, edgecolor='white',
                    linewidth=0.5, alpha=0.85)

for bar, val in zip(bars, segment_cltv.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${val:,.0f}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')

axes[1].set_title('Average 12-Month CLTV by Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Customer Segment', fontsize=11)
axes[1].set_ylabel('Average 12-Month CLTV ($)', fontsize=11)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Customer Segmentation Analysis — Infotact Solutions',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/05_customer_segments.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Customer segment charts saved as outputs/05_customer_segments.png")

## Step 32: AOV by Cohort Bar Chart
Comparing the Average Order Value across different acquisition cohorts
to identify which cohorts have the highest spending customers.

In [ ]:
# AOV by cohort
cohort_aov = cltv_df.groupby('CohortMonth')['AOV'].mean().reset_index()
cohort_aov['CohortMonth'] = cohort_aov['CohortMonth'].astype(str)
cohort_aov = cohort_aov.sort_values('AOV', ascending=False)

plt.figure(figsize=(16, 7))
bar_colors = ['#C00000' if i == 0 else '#1F4E79' for i in range(len(cohort_aov))]
bars = plt.bar(cohort_aov['CohortMonth'], cohort_aov['AOV'],
               color=bar_colors, edgecolor='white',
               linewidth=0.5, alpha=0.85)

plt.title('Average Order Value (AOV) by Acquisition Cohort\nSaaS/E-Commerce Analysis — Infotact Solutions',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cohort Month', fontsize=11)
plt.ylabel('Average Order Value ($)', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.axhline(y=cohort_aov['AOV'].mean(), color='red',
            linestyle='--', linewidth=1.5,
            label=f'Overall Average AOV: ${cohort_aov["AOV"].mean():,.0f}')
plt.legend(fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/06_aov_by_cohort.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ AOV by cohort chart saved as outputs/06_aov_by_cohort.png")

## Step 33: Save All Key Outputs
Saving all key dataframes as CSV files locally for reference.

> ⚠️ CSV files are excluded from GitHub via .gitignore

In [ ]:
os.makedirs('outputs', exist_ok=True)

# Save percentage retention matrix
percentage_matrix.to_csv('outputs/percentage_retention_matrix.csv')
print("✅ Percentage retention matrix saved")

# Save CLTV dataframe
cltv_df.to_csv('outputs/cltv_summary.csv', index=False)
print("✅ CLTV summary saved")

# Save cohort CLTV summary
cohort_cltv.to_csv('outputs/cohort_cltv_summary.csv', index=False)
print("✅ Cohort CLTV summary saved")

print(f"\n✅ All outputs saved to outputs/ folder")
print("\nVisualization Files Produced:")
print("  01_cohort_retention_heatmap.png")
print("  02_retention_decay_charts.png")
print("  03_revenue_by_cohort.png")
print("  04_top_countries_cltv.png")
print("  05_customer_segments.png")
print("  06_aov_by_cohort.png")

---
## 🎯 Project Summary & Strategic Recommendations

### ✅ All Steps Completed:
- **Week 1:** Data cleaned — valid transactions from 541,909 raw records
- **Week 2:** Retention matrix built — 79.38% Month 1 churn identified
- **Week 3:** CLTV calculated across 4,338 unique customers
- **Week 4:** 6 visualizations produced — heatmap, decay charts, revenue and segment analysis

### 📊 Key Business Findings Summary:

| Metric | Value |
|---|---|
| Overall Month 1 Retention | 20.62% |
| Month 0 to Month 1 Drop | -79.38% |
| Best Performing Cohort | December 2010 (36.61%) |
| Worst Performing Cohort | November 2011 (11.15%) |
| Month 12 Average Retention | 26.55% |
| High Value Customer 12-Month CLTV | \$77,835 |
| Mid Value Customer 12-Month CLTV | \$9,168 |
| Low Value Customer 12-Month CLTV | \$2,136 |
| High Value Customer Max CAC | \$23,350 |
| Mid Value Customer Max CAC | \$2,750 |
| Low Value Customer Max CAC | \$640 |
| Top Country by CLTV | Netherlands \$246,720 |
| Total Revenue Cross-Validation | \$8,887,208.89 — Match: True |

### 🎯 Top 7 Strategic Recommendations:

**Priority 1 — Fix Early Churn (Most Urgent)**
> 79.38% of customers churn after their first purchase.
> Implement an automated re-engagement email sequence triggered
> 7 days after first purchase with a discount incentive.
> Target: Reduce Month 1 churn by 15-20%.

**Priority 2 — Protect High Value Customers**
> Only 25% of customers (1,085) are High Value but generate
> \$77,835 average 12-Month CLTV — nearly 9x more than Mid Value.
> Implement a VIP loyalty program for this segment immediately.

**Priority 3 — Optimize CAC by Segment**
> Never spend more than **\$23,350** acquiring a High Value customer.
> Never spend more than **\$640** acquiring a Low Value customer.
> Set separate acquisition budgets per segment to avoid marketing waste.

**Priority 4 — Geographic Expansion**
> Netherlands **(\$246,720)** and EIRE **(\$135,489)** customers generate
> significantly higher CLTV than the United Kingdom **(\$7,780)**.
> Increase marketing investment in these high-value markets.

**Priority 5 — Seasonal Campaign Strategy**
> December 2010 cohort is the best performing at 36.61% Month 1 retention.
> Maximise acquisition budget in November-December every year
> to acquire the highest quality retention cohorts.

**Priority 6 — 30-Day Onboarding Sequence**
> Implement structured 30-day onboarding with weekly email touchpoints
> to guide new customers through their critical first 4 weeks.
> A customer retained past Month 1 is significantly more likely
> to become a long-term loyal customer.

**Priority 7 — Month 7 Re-engagement Campaign**
> Data shows a consistent downward fluctuation at Month 7.
> Implement a targeted re-engagement campaign at Month 7
> with exclusive offers to prevent this predictable churn point.

---
*Master Notebook Complete — Infotact Solutions Data Analytics Internship Project 2*
*All cells run sequentially top to bottom without errors.*
*Kernel — Restart and Clear Output before final commit.*